# Explore
Explore what subreddits relate to or mention resumes.

In [2]:
from dotenv import load_dotenv
import os

In [3]:
load_dotenv()

True

In [4]:
client_id = os.environ.get("REDDIT_CLIENT_ID")
client_secret = os.environ.get("REDDIT_CLIENT_SECRET")

In [5]:
import praw

client= praw.Reddit(
    client_id=client_id,
    client_secret=client_secret,
    user_agent="research/0.1"
)

## Exploring Subreddits
Found that r/resumes is a prominent subreddit

In [6]:
resume_subs = client.subreddits.search_by_name("resume")

for sub in resume_subs:
    print(f"r/{sub.display_name} - {sub.subscribers:,} subscribers")

r/Resume - 130,812 subscribers
r/resumes - 1,328,408 subscribers
r/ResumeExperts - 29,598 subscribers
r/ResumeTips - 18,118 subscribers
r/ResumeCoverLetterTips - 27,848 subscribers
r/resumesindia - 7,321 subscribers
r/resumesupport - 8,767 subscribers
r/ResumeFairies - 7,082 subscribers
r/resumeph - 16 subscribers
r/resumereview - 2,295 subscribers


## Downloading sample of r/resumes Posts
Retrieved all posts with >=1 comment over time period spanning January 1st, 2026 to November 1st, 2026.

In [13]:
import deepsift

In [38]:
from deepsift.scraper import RedditScraper
from deepsift.filters import Filters
from deepsift.models import PostField

filters = Filters(
    post_include_fields=[
        PostField.ID,
        PostField.TITLE,
        PostField.SELFTEXT,
        PostField.SCORE,
        PostField.NUM_COMMENTS,
        PostField.CREATED_UTC,
        PostField.SUBREDDIT,
        PostField.URL
    ],
    post_filter=lambda post: post.num_comments >= 1,
)

scraper = RedditScraper(
    filters=filters,
    post_workers=4,
    shard_days=30,
)

# uncomment to run extraction
# post_ids = scraper.scrape_posts(
#     subreddits=["resumes"],
#     from_dt="2026-01-01",
#     to_dt="2026-11-01",
#     output_path="data/subreddit_resumes_posts.parquet",
# )

# print(f"Scraped {len(post_ids)} curated posts.")

## Finding Relevant Posts in Retrieved Data

In [58]:
import pandas as pd

posts_path = "data/subreddit_resumes_posts.parquet"
posts = pd.read_parquet(posts_path)

In [59]:
posts.shape

(20827, 8)

In [60]:
# validating the time period was respected
posts["created_utc"] = pd.to_datetime(posts["created_utc"], unit="s")

In [61]:
# this validates that it we did get data for the complete time period requested
posts["created_utc"].describe()

count                         20827
mean     2026-05-14 14:14:26.002256
min             2026-01-01 00:29:21
25%      2026-02-25 15:13:13.500000
50%             2026-05-04 21:39:57
75%             2026-07-28 12:10:49
max             2026-10-31 22:55:34
Name: created_utc, dtype: object

In [81]:
# dropping duplicates to prevent double processing
clean_posts = posts.drop_duplicates(subset=["selftext"])

In [82]:
# exploring values of selftext
freq_text = pd.DataFrame(posts["selftext"].value_counts())

In [88]:
# removing posts that have been removed or are empty
clean_posts = clean_posts[
    (clean_posts["selftext"] != "[removed]")
    & (clean_posts["selftext"] != "[deleted]")
    & (clean_posts["selftext"] != "")
]

In [90]:
clean_posts.head(1)

,created_utc,id,num_comments,score,selftext,subreddit,title,url
3,2026-09-01 22:54:11,1w4sng0,3,3,"I'm know all my ""professional experience"" is a...",resumes,"[4 YoE, Sub Teacher, Remote/USA, Looking to Pi...",https://i.redd.it/5q7fhepjnzmh1.png


In [93]:
# checking distribution of number of comments
clean_posts["num_comments"].describe()

count    9506.000000
mean        4.592889
std         8.100685
min         1.000000
25%         1.000000
50%         3.000000
75%         5.000000
max       271.000000
Name: num_comments, dtype: float64

In [94]:
# checking distribution of score associated with posts
clean_posts["score"].describe()

count    9506.000000
mean        2.981380
std        21.133572
min         0.000000
25%         1.000000
50%         1.000000
75%         2.000000
max      1175.000000
Name: score, dtype: float64

In [106]:
# checking the extension associated with url of post
endings = pd.DataFrame(clean_posts["url"].str.split(".").str[-1].str.lower().value_counts())
likely_extensions = endings[endings.index.str.len() < 10]
likely_extensions

,count
url,
jpeg,2110
png,1678
0,1
cv,1
pro/,1
me,1
app/,1
com,1


In [ ]:
# filtering posts to only ones with image attaches
has_attachment = clean_posts["url"].str.split(".").str[-1].str.lower().isin(["jpeg", "png"])
clean_posts = clean_posts[has_attachment]

In [159]:
# noticed a common regex pattern in titles of posts
import re

sample_title = clean_posts["title"].iloc[0]
res = re.search(r"\[(.*)\]", sample_title)
res.group(1)

clean_posts["resume_type"] = clean_posts["title"].str.extract(r"\[(.*)\]")
clean_posts.dropna(inplace=True)

/var/folders/my/90dxhc_500vbg9nf8jbt4lq80000gn/T/ipykernel_1940/1603955904.py:9: Pandas4Warning: The inplace keyword in DataFrame.dropna is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  clean_posts.dropna(inplace=True)


In [160]:
clean_posts["resume_type"].value_counts(dropna=False)

resume_type
2 YoE, Unemployed, Digital Marketing, USA                                          7
0 YoE, Unemployed, Retail Assistant, United Kingdom                                5
0 YoE, Unemployed, Artificial Intelligence Engineer, India                         4
0 YoE, Unemployed, Retail Stocker, United States                                   4
0 YoE, Unemployed, Software Engineer, India                                        4
                                                                                  ..
1st Year University Student, intern, Business Field, United State                  1
0 YOE, Data Analyst Co-op, Data Science Co-op, Canada                              1
3 years, intern, intern, USA                                                       1
7 YOE, Employed, System Administrator/Cloud Administrator, USA                     1
1 YoE, IT Business Analyst Intern, Associate/Jr Business Analyst, United States    1
Name: count, Length: 3579, dtype: int64

In [161]:
# 97% of the clean posts have this somewhere in the title, so we can use this to categorize the posts
clean_posts["resume_type"].notna().sum() / clean_posts.shape[0]

np.float64(1.0)

In [164]:
# breaking up the resume type into more granular categories
sample_resume_type = clean_posts["resume_type"].iloc[20]
sample_resume_type.split(",")


['3 YoE', ' Graduate Student', ' MaCC', ' United States']

In [163]:
clean_posts["resume_type"].iloc[20]

'3 YoE, Graduate Student, MaCC, United States'

In [183]:
split_cols = clean_posts["resume_type"].str.split(",", expand=True)

In [167]:
split_cols.info()

<class 'pandas.DataFrame'>
Index: 3712 entries, 3 to 20826
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   0       3712 non-null   str  
 1   1       3712 non-null   str  
 2   2       3712 non-null   str  
 3   3       3707 non-null   str  
 4   4       175 non-null    str  
 5   5       30 non-null     str  
 6   6       5 non-null      str  
 7   7       2 non-null      str  
dtypes: str(8)
memory usage: 459.5 KB


In [187]:
split_cols[pd.isna(split_cols[7]) == False]

,0,1,2,3,4,5,6,7
670,1 YoE,Recent Graduate,Entry Level Supply Chain Management (Operations,Logistics,Data Analyst,Procurement,Etc.),USA
16813,10 YoE,unemployed,growth,demand,lifecycle,direct,manager/director marketing role,USA


In [198]:
split_cols[pd.isna(split_cols[3]) == False][0].value_counts().head(80)

0
0 YoE      859
1 YoE      275
2 YoE      238
3 YoE      227
0 years    173
          ... 
6YoE         4
5 Yoe        4
3 Years      4
4 Years      4
10+ YoE      4
Name: count, Length: 80, dtype: int64